# Visualizando la loss de redes neuronales

Vamos a utilizar el framework [Loss Landscape](https://github.com/iturbideM/loss-landscape/) del paper "Visualizing the Loss Landscape of Neural Nets" para visualizar la superficie de la loss de nuestra red en 2 direcciónes aleatorias.

In [ ]:
!git clone https://github.com/iturbideM/loss-landscape.git

In [ ]:
!pip install h5py matplotlib seaborn scikit-learn mpi4py

## Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
import os
import sys
sys.path.append("loss-landscape")

## VGG9

In [ ]:
from cifar10.models.vgg import VGG9

transform = transforms.Compose([transforms.ToTensor()])
trainset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True)

model = VGG9()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
model.to(device)

optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
criterion = nn.CrossEntropyLoss()

epochs = 10

for i in range(epochs):
  model.train()
  train_loss = 0
  for inputs, targets in trainloader:
      inputs, targets = inputs.to(device), targets.to(device)
      optimizer.zero_grad()
      outputs = model(inputs)
      loss = criterion(outputs, targets)
      loss.backward()
      optimizer.step()

      train_loss += loss.item()

  train_loss /= len(trainloader)
  print(f"Epoch {i+1}/{epochs}, Loss: {train_loss}")

# Save model
os.makedirs("trained_models", exist_ok=True)
torch.save(model.state_dict(), "trained_models/vgg_weights.pth")


In [ ]:
!python loss-landscape/plot_surface.py \
  --cuda \
  --dataset cifar10 \
  --x=-1:1:26 \
  --y=-1:1:26 \
  --dir_type weights \
  --xnorm filter \
  --ynorm filter \
  --xignore biasbn \
  --yignore biasbn \
  --data_split 100 \
  --idx 0\
  --vmax 150\
  --model_file trained_models/vgg_weights.pth \
  --model vgg9 \
  --surf_file vgg_9_surface.h5

In [ ]:
import plot_2D

plot_2D.plot_2d_contour(surf_file='vgg_9_surface.h5', vlevel=0.1, show=True)

---

Podemos generar un archivo en formato de **ParaView** que podemos cargar en un software como [Kitware](https://kitware.github.io/glance/app/) para visualizarlos

In [ ]:
!python loss-landscape/h52vtp.py --surf_file vgg_9_surface.h5 --surf_name train_loss --zmax  10 --log

## Ejemplo de red para cifar10

Si queremos hacer nuestra propia red, debemos crearnos un archivo nuevo, por ejemplo `my_models.py` y definir la arquitectura de nuestros modelos ahi dentro. Por ejemplo podemos definirnos una red muy simple que haga clasificación de imágenes utilizando el dataset cifar10

```python
import torch
import torch.nn as nn

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, 3, padding=1)
        self.relu = nn.ReLU()
        self.fc = nn.Linear(64*32*32, 10)
    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = x.view(x.size(0), -1)
        return self.fc(x)
```

### Entrenando la red

In [ ]:
from my_models import SimpleCNN

In [ ]:
transform = transforms.Compose([transforms.ToTensor()])
trainset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True)

model = SimpleCNN()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
criterion = nn.CrossEntropyLoss()

epochs = 10

for i in range(epochs):
  model.train()
  train_loss = 0
  for inputs, targets in trainloader:
      inputs, targets = inputs.to(device), targets.to(device)
      optimizer.zero_grad()
      outputs = model(inputs)
      loss = criterion(outputs, targets)
      loss.backward()
      optimizer.step()

      train_loss += loss.item()

  train_loss /= len(trainloader)
  print(f"Epoch {i+1}/{epochs}, Loss: {train_loss}")

# Save model
os.makedirs("trained_models", exist_ok=True)
torch.save(model.state_dict(), "trained_models/simplecnn.pth")


In [ ]:
!python loss-landscape/plot_surface.py \
  --cuda \
  --dataset cifar10 \
  --x=-1:1:26 \
  --y=-1:1:26 \
  --dir_type weights \
  --xnorm filter \
  --ynorm filter \
  --xignore biasbn \
  --yignore biasbn \
  --data_split 100 \
  --idx 1\
  --vmax 150\
  --model_file trained_models/simplevgg.pth \
  --model custom:my_models.SimpleCNN \
  --surf_file simplecnn_surface.h5

In [ ]:
import plot_2D

plot_2D.plot_2d_contour(surf_file='simplecnn_surface.h5', vlevel=0.01, show=True)